In [96]:
%pip install pandas
%pip install openpyxl
%pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.


Note: you may need to restart the kernel to use updated packages.


  Using cached scikit_learn-1.7.2-cp310-cp310-win_amd64.whl.metadata (11 kB)
Using cached scikit_learn-1.7.2-cp310-cp310-win_amd64.whl (8.9 MB)
   ---------------------------------------- 0.0/41.3 MB ? eta -:--:--
   - -------------------------------------- 1.3/41.3 MB 7.4 MB/s eta 0:00:06
   --- ------------------------------------ 3.4/41.3 MB 8.7 MB/s eta 0:00:05
   ----- ---------------------------------- 5.5/41.3 MB 9.1 MB/s eta 0:00:04
   ------ --------------------------------- 6.8/41.3 MB 8.7 MB/s eta 0:00:04
   -------- ------------------------------- 8.9/41.3 MB 8.8 MB/s eta 0:00:04
   ---------- ----------------------------- 10.7/41.3 MB 8.7 MB/s eta 0:00:04
   ----------- ---------------------------- 12.3/41.3 MB 8.6 MB/s eta 0:00:04
   ------------ --------------------------- 13.4/41.3 MB 8.1 MB/s eta 0:00:04
   -------------- ------------------------- 14.9/41.3 MB 8.0 MB/s eta 0:00:04
   --------------- ------------------------ 16.0/41.3 MB 7.9 MB/s eta 0:00:04
   --------

In [2]:
import pandas as pd
import numpy as np
import os
import openpyxl
import csv

In [3]:


excel = openpyxl.load_workbook("./raw/fatal_overdose.xlsx")
sheet = excel['Data']

with open("./raw/fatal_overdose.csv", 'w', newline="") as f:
    writer = csv.writer(f)

    for row in sheet.rows:
        writer.writerow([cell.value for cell in row])

df = pd.DataFrame(pd.read_csv("./raw/fatal_overdose.csv"))

print(df)

      Jurisdiction  year  alldrug_deaths  alldrug_rate  opioids_deaths  \
0           Alaska  2020             171          23.8             111   
1          Arizona  2020            2412          34.0            1867   
2         Colorado  2020            1329          22.4             976   
3      Connecticut  2020            1374          39.4            1278   
4         Delaware  2020             446          47.4             417   
..             ...   ...             ...           ...             ...   
187       Virginia  2024            1559          17.9            1181   
188     Washington  2024            2988          37.0            2470   
189  West Virginia  2024             782          46.3             611   
190      Wisconsin  2024            1054          18.1             783   
191        Wyoming  2024              95          16.6              52   

     opioids_rate  imfs_deaths  imfs_rate  heroin_deaths  heroin_rate  ...  \
0            15.4           58   

In [5]:
numeric_df = df.select_dtypes(include=np.number).copy()
categorical_df = df.select_dtypes(exclude=np.number).copy()

In [ ]:
mask = categorical_df.isin(['<0.1']).any(axis=1)

# Keep only the rows where the mask is False
df_clean = categorical_df[~mask]
df_clean

In [62]:
df_clean.dtypes

Jurisdiction                       object
combo1_drugs                       object
combo2_drugs                       object
combo3_drugs                       object
combo4_drugs                       object
combo5_drugs                       object
carfentanil_percent                object
parafluorofentanyl_percent         object
other_illegal_opioids_percent      object
nitazenes_percent                  object
isotonitazene_percent              object
metonitazene_percent               object
adulterants_sedatives_percent      object
xylazine_percent                   object
gabapentin_percent                 object
illegal_benzodiazepines_percent    object
bromazolam_percent                 object
etizolam_percent                   object
flualprazolam_percent              object
ketamine_percent                   object
kratom_percent                     object
synthetic_cannabinoid_percent      object
cathinones_percent                 object
aian_nh_percent                   

In [111]:
'''

##################################

'''
perc_obj_rows = df_clean.drop(df_clean[['Jurisdiction','combo1_drugs','combo2_drugs','combo3_drugs','combo4_drugs','combo5_drugs']], axis=1)
perc_df = perc_obj_rows.astype(float)

# Replace 9999.0 with NaN, then fill with the mean of each column
perc_df = perc_df.replace(9999.0, np.nan).fillna(perc_df[perc_df != 9999.0].mean())


obj_rows = df_clean.drop(perc_df, axis=1)
obj_rows.nunique()

Jurisdiction    40
combo1_drugs     5
combo2_drugs     7
combo3_drugs     9
combo4_drugs    16
combo5_drugs    20
dtype: int64

In [120]:
import pandas as pd
import numpy as np

obj_rows['target'] = None
# 1. Calculate the mean target value for each drug
# We melt the dataframe to see every instance of a drug regardless of column
melted = obj_rows.melt(id_vars=['target'], value_vars=['combo1_drugs', 'combo2_drugs', 'combo3_drugs', 'combo4_drugs', 'combo5_drugs'])
drug_means = melted.groupby('value')['target'].mean().to_dict()

# 2. Define your weights (Column 1 is most important)
# You can use linear decay, exponential decay, or custom weights
weights = {
    'combo1_drugs': 1.0,
    'combo2_drugs': 0.8,
    'combo3_drugs': 0.6,
    'combo4_drugs': 0.4,
    'combo5_drugs': 0.2
}

# 3. Apply the encoding and the weights
encoded_df = obj_rows.copy()

for col, weight in weights.items():
    # Replace string with its mean target value, then multiply by the importance weight
    encoded_df[f'{col}_weighted'] = encoded_df[col].map(drug_means) * weight

# 4. Create a final aggregate feature
weighted_cols = [f'{col}_weighted' for col in weights.keys()]
encoded_df['total_combo_score'] = encoded_df[weighted_cols].sum(axis=1)

In [121]:
encoded_df

,Jurisdiction,combo1_drugs,combo2_drugs,combo3_drugs,combo4_drugs,combo5_drugs,target,combo1_drugs_weighted,combo2_drugs_weighted,combo3_drugs_weighted,combo4_drugs_weighted,combo5_drugs_weighted,total_combo_score
0,Alaska,"METHAMPHETAMINE, NO OTHER STIMULANTS OR OPIOIDS","ILLEGALLY-MADE FENTANYLS, NO OTHER OPIOIDS OR ...","PRESCRIPTION OPIOIDS, NO OTHER OPIOIDS OR STIM...","HEROIN, PRESCRIPTION OPIOIDS, AND METHAMPHETAMINE",HEROIN AND METHAMPHETAMINE,None,NaN,NaN,NaN,NaN,NaN,0.0
2,Colorado,"ILLEGALLY-MADE FENTANYLS, NO OTHER OPIOIDS OR ...","METHAMPHETAMINE, NO OTHER STIMULANTS OR OPIOIDS","PRESCRIPTION OPIOIDS, NO OTHER OPIOIDS OR STIM...",ILLEGALLY-MADE FENTANYLS AND COCAINE,"HEROIN, PRESCRIPTION OPIOIDS, AND METHAMPHETAMINE",None,NaN,NaN,NaN,NaN,NaN,0.0
3,Connecticut,"ILLEGALLY-MADE FENTANYLS, NO OTHER OPIOIDS OR ...",ILLEGALLY-MADE FENTANYLS AND COCAINE,ILLEGALLY-MADE FENTANYLS AND PRESCRIPTION OPIO...,"ILLEGALLY-MADE FENTANYLS AND HEROIN, NO STIMUL...","ILLEGALLY-MADE FENTANYLS, HEROIN, AND COCAINE",None,NaN,NaN,NaN,NaN,NaN,0.0
4,Delaware,"ILLEGALLY-MADE FENTANYLS, NO OTHER OPIOIDS OR ...",ILLEGALLY-MADE FENTANYLS AND COCAINE,"ILLEGALLY-MADE FENTANYLS AND HEROIN, NO STIMUL...","ILLEGALLY-MADE FENTANYLS, HEROIN, AND COCAINE","PRESCRIPTION OPIOIDS, NO OTHER OPIOIDS OR STIM...",None,NaN,NaN,NaN,NaN,NaN,0.0
5,District of Columbia,"ILLEGALLY-MADE FENTANYLS, NO OTHER OPIOIDS OR ...",ILLEGALLY-MADE FENTANYLS AND COCAINE,"ILLEGALLY-MADE FENTANYLS AND HEROIN, NO STIMUL...","COCAINE, NO OTHER STIMULANTS OR OPIOIDS","ILLEGALLY-MADE FENTANYLS, HEROIN, AND COCAINE",None,NaN,NaN,NaN,NaN,NaN,0.0
6,Georgia,"METHAMPHETAMINE, NO OTHER STIMULANTS OR OPIOIDS","ILLEGALLY-MADE FENTANYLS, NO OTHER OPIOIDS OR ...","PRESCRIPTION OPIOIDS, NO OTHER OPIOIDS OR STIM...","COCAINE, NO OTHER STIMULANTS OR OPIOIDS","ILLEGALLY-MADE FENTANYLS AND HEROIN, NO STIMUL...",None,NaN,NaN,NaN,NaN,NaN,0.0
7,Iowa,"METHAMPHETAMINE, NO OTHER STIMULANTS OR OPIOIDS","ILLEGALLY-MADE FENTANYLS, NO OTHER OPIOIDS OR ...","ILLEGALLY-MADE FENTANYLS AND HEROIN, NO STIMUL...",ILLEGALLY-MADE FENTANYLS AND METHAMPHETAMINE,"PRESCRIPTION OPIOIDS, NO OTHER OPIOIDS OR STIM...",None,NaN,NaN,NaN,NaN,NaN,0.0
8,Kansas,"METHAMPHETAMINE, NO OTHER STIMULANTS OR OPIOIDS","ILLEGALLY-MADE FENTANYLS, NO OTHER OPIOIDS OR ...","PRESCRIPTION OPIOIDS, NO OTHER OPIOIDS OR STIM...",ILLEGALLY-MADE FENTANYLS AND METHAMPHETAMINE,"COCAINE, NO OTHER STIMULANTS OR OPIOIDS",None,NaN,NaN,NaN,NaN,NaN,0.0
10,Maine,"ILLEGALLY-MADE FENTANYLS, NO OTHER OPIOIDS OR ...",ILLEGALLY-MADE FENTANYLS AND COCAINE,"PRESCRIPTION OPIOIDS, NO OTHER OPIOIDS OR STIM...",ILLEGALLY-MADE FENTANYLS AND METHAMPHETAMINE,ILLEGALLY-MADE FENTANYLS AND PRESCRIPTION OPIO...,None,NaN,NaN,NaN,NaN,NaN,0.0
14,Minnesota,"ILLEGALLY-MADE FENTANYLS, NO OTHER OPIOIDS OR ...","METHAMPHETAMINE, NO OTHER STIMULANTS OR OPIOIDS",ILLEGALLY-MADE FENTANYLS AND METHAMPHETAMINE,"PRESCRIPTION OPIOIDS, NO OTHER OPIOIDS OR STIM...","ILLEGALLY-MADE FENTANYLS AND HEROIN, NO STIMUL...",None,NaN,NaN,NaN,NaN,NaN,0.0


In [118]:
obj_rows

,Jurisdiction,combo1_drugs,combo2_drugs,combo3_drugs,combo4_drugs,combo5_drugs
0,Alaska,"METHAMPHETAMINE, NO OTHER STIMULANTS OR OPIOIDS","ILLEGALLY-MADE FENTANYLS, NO OTHER OPIOIDS OR ...","PRESCRIPTION OPIOIDS, NO OTHER OPIOIDS OR STIM...","HEROIN, PRESCRIPTION OPIOIDS, AND METHAMPHETAMINE",HEROIN AND METHAMPHETAMINE
2,Colorado,"ILLEGALLY-MADE FENTANYLS, NO OTHER OPIOIDS OR ...","METHAMPHETAMINE, NO OTHER STIMULANTS OR OPIOIDS","PRESCRIPTION OPIOIDS, NO OTHER OPIOIDS OR STIM...",ILLEGALLY-MADE FENTANYLS AND COCAINE,"HEROIN, PRESCRIPTION OPIOIDS, AND METHAMPHETAMINE"
3,Connecticut,"ILLEGALLY-MADE FENTANYLS, NO OTHER OPIOIDS OR ...",ILLEGALLY-MADE FENTANYLS AND COCAINE,ILLEGALLY-MADE FENTANYLS AND PRESCRIPTION OPIO...,"ILLEGALLY-MADE FENTANYLS AND HEROIN, NO STIMUL...","ILLEGALLY-MADE FENTANYLS, HEROIN, AND COCAINE"
4,Delaware,"ILLEGALLY-MADE FENTANYLS, NO OTHER OPIOIDS OR ...",ILLEGALLY-MADE FENTANYLS AND COCAINE,"ILLEGALLY-MADE FENTANYLS AND HEROIN, NO STIMUL...","ILLEGALLY-MADE FENTANYLS, HEROIN, AND COCAINE","PRESCRIPTION OPIOIDS, NO OTHER OPIOIDS OR STIM..."
5,District of Columbia,"ILLEGALLY-MADE FENTANYLS, NO OTHER OPIOIDS OR ...",ILLEGALLY-MADE FENTANYLS AND COCAINE,"ILLEGALLY-MADE FENTANYLS AND HEROIN, NO STIMUL...","COCAINE, NO OTHER STIMULANTS OR OPIOIDS","ILLEGALLY-MADE FENTANYLS, HEROIN, AND COCAINE"
6,Georgia,"METHAMPHETAMINE, NO OTHER STIMULANTS OR OPIOIDS","ILLEGALLY-MADE FENTANYLS, NO OTHER OPIOIDS OR ...","PRESCRIPTION OPIOIDS, NO OTHER OPIOIDS OR STIM...","COCAINE, NO OTHER STIMULANTS OR OPIOIDS","ILLEGALLY-MADE FENTANYLS AND HEROIN, NO STIMUL..."
7,Iowa,"METHAMPHETAMINE, NO OTHER STIMULANTS OR OPIOIDS","ILLEGALLY-MADE FENTANYLS, NO OTHER OPIOIDS OR ...","ILLEGALLY-MADE FENTANYLS AND HEROIN, NO STIMUL...",ILLEGALLY-MADE FENTANYLS AND METHAMPHETAMINE,"PRESCRIPTION OPIOIDS, NO OTHER OPIOIDS OR STIM..."
8,Kansas,"METHAMPHETAMINE, NO OTHER STIMULANTS OR OPIOIDS","ILLEGALLY-MADE FENTANYLS, NO OTHER OPIOIDS OR ...","PRESCRIPTION OPIOIDS, NO OTHER OPIOIDS OR STIM...",ILLEGALLY-MADE FENTANYLS AND METHAMPHETAMINE,"COCAINE, NO OTHER STIMULANTS OR OPIOIDS"
10,Maine,"ILLEGALLY-MADE FENTANYLS, NO OTHER OPIOIDS OR ...",ILLEGALLY-MADE FENTANYLS AND COCAINE,"PRESCRIPTION OPIOIDS, NO OTHER OPIOIDS OR STIM...",ILLEGALLY-MADE FENTANYLS AND METHAMPHETAMINE,ILLEGALLY-MADE FENTANYLS AND PRESCRIPTION OPIO...
14,Minnesota,"ILLEGALLY-MADE FENTANYLS, NO OTHER OPIOIDS OR ...","METHAMPHETAMINE, NO OTHER STIMULANTS OR OPIOIDS",ILLEGALLY-MADE FENTANYLS AND METHAMPHETAMINE,"PRESCRIPTION OPIOIDS, NO OTHER OPIOIDS OR STIM...","ILLEGALLY-MADE FENTANYLS AND HEROIN, NO STIMUL..."


In [94]:
obj_rows.combo3_drugs.unique()

array(['PRESCRIPTION OPIOIDS, NO OTHER OPIOIDS OR STIMULANTS',
       'ILLEGALLY-MADE FENTANYLS AND PRESCRIPTION OPIOIDS, NO STIMULANTS',
       'ILLEGALLY-MADE FENTANYLS AND HEROIN, NO STIMULANTS',
       'ILLEGALLY-MADE FENTANYLS AND METHAMPHETAMINE',
       'HEROIN AND METHAMPHETAMINE',
       'ILLEGALLY-MADE FENTANYLS AND COCAINE',
       'ILLEGALLY-MADE FENTANYLS, NO OTHER OPIOIDS OR STIMULANTS',
       'COCAINE, NO OTHER STIMULANTS OR OPIOIDS',
       'METHAMPHETAMINE, NO OTHER STIMULANTS OR OPIOIDS'], dtype=object)

In [97]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder

# 1. Get all unique values across the entire slice of the dataframe
all_values = pd.concat([df['combo1_drugs'], df['combo2_drugs'], df['combo3_drugs'], df['combo4_drugs'], df['combo5_drugs']]).unique()

# 2. Fit one encoder for everything
le = LabelEncoder()
le.fit(all_values)

# 3. Transform all columns using the same encoder
for col in ['combo1_drugs','combo2_drugs','combo3_drugs','combo4_drugs','combo5_drugs']:
    df[col] = le.transform(df[col])

In [98]:
all_values

array(['METHAMPHETAMINE, NO OTHER STIMULANTS OR OPIOIDS',
       'ILLEGALLY-MADE FENTANYLS, NO OTHER OPIOIDS OR STIMULANTS',
       'ILLEGALLY-MADE FENTANYLS AND COCAINE',
       'PRESCRIPTION OPIOIDS, NO OTHER OPIOIDS OR STIMULANTS',
       'ILLEGALLY-MADE FENTANYLS AND METHAMPHETAMINE',
       'ILLEGALLY-MADE FENTANYLS, HEROIN, AND METHAMPHETAMINE',
       'COCAINE, NO OTHER STIMULANTS OR OPIOIDS',
       'ILLEGALLY-MADE FENTANYLS AND PRESCRIPTION OPIOIDS, NO STIMULANTS',
       'ILLEGALLY-MADE FENTANYLS AND HEROIN, NO STIMULANTS',
       'HEROIN AND METHAMPHETAMINE',
       'HEROIN, PRESCRIPTION OPIOIDS, AND METHAMPHETAMINE',
       'ILLEGALLY-MADE FENTANYLS, HEROIN, AND COCAINE',
       'HEROIN, NO OTHER OPIOIDS OR STIMULANTS',
       'ILLEGALLY-MADE FENTANYLS, HEROIN, PRESCRIPTION OPIOIDS, AND METHAMPHETAMINE',
       'ILLEGALLY-MADE FENTANYLS, COCAINE, AND METHAMPHETAMINE',
       'ILLEGALLY-MADE FENTANYLS, PRESCRIPTION OPIOIDS, AND COCAINE',
       'HEROIN AND PRESCRIPTION OPIOI

In [93]:
obj_rows.combo5_drugs.unique()

array(['HEROIN AND METHAMPHETAMINE',
       'HEROIN, PRESCRIPTION OPIOIDS, AND METHAMPHETAMINE',
       'ILLEGALLY-MADE FENTANYLS, HEROIN, AND COCAINE',
       'PRESCRIPTION OPIOIDS, NO OTHER OPIOIDS OR STIMULANTS',
       'ILLEGALLY-MADE FENTANYLS AND HEROIN, NO STIMULANTS',
       'COCAINE, NO OTHER STIMULANTS OR OPIOIDS',
       'ILLEGALLY-MADE FENTANYLS AND PRESCRIPTION OPIOIDS, NO STIMULANTS',
       'ILLEGALLY-MADE FENTANYLS, NO OTHER OPIOIDS OR STIMULANTS',
       'ILLEGALLY-MADE FENTANYLS AND METHAMPHETAMINE',
       'HEROIN AND PRESCRIPTION OPIOIDS, NO STIMULANTS',
       'ILLEGALLY-MADE FENTANYLS AND COCAINE',
       'HEROIN, NO OTHER OPIOIDS OR STIMULANTS',
       'ILLEGALLY-MADE FENTANYLS, PRESCRIPTION OPIOIDS, AND COCAINE',
       'METHAMPHETAMINE, NO OTHER STIMULANTS OR OPIOIDS',
       'ILLEGALLY-MADE FENTANYLS, HEROIN, AND METHAMPHETAMINE',
       'ILLEGALLY-MADE FENTANYLS, PRESCRIPTION OPIOIDS, AND METHAMPHETAMINE',
       'ILLEGALLY-MADE FENTANYLS, COCAINE, AND METHAM

In [45]:
pd.set_option('display.max_rows', None)
df_clean.dtypes

Jurisdiction                        object
year                                 int64
alldrug_deaths                       int64
alldrug_rate                       float64
opioids_deaths                       int64
opioids_rate                       float64
imfs_deaths                          int64
imfs_rate                          float64
heroin_deaths                        int64
heroin_rate                        float64
rxopioids_deaths                     int64
rxopioids_rate                     float64
stimulant_deaths                     int64
stimulant_rate                     float64
cocaine_deaths                       int64
cocaine_rate                       float64
meth_deaths                          int64
meth_rate                          float64
nonopioid_sedatives_deaths           int64
nonopioid_sedatives_rate           float64
benzodiazepines_deaths               int64
benzodiazepines_rate               float64
opioids_percent                    float64
imfs_percen

In [48]:
categorical_df.dtypes

Jurisdiction                       object
combo1_drugs                       object
combo2_drugs                       object
combo3_drugs                       object
combo4_drugs                       object
combo5_drugs                       object
carfentanil_percent                object
parafluorofentanyl_percent         object
other_illegal_opioids_percent      object
nitazenes_percent                  object
isotonitazene_percent              object
metonitazene_percent               object
adulterants_sedatives_percent      object
xylazine_percent                   object
gabapentin_percent                 object
illegal_benzodiazepines_percent    object
bromazolam_percent                 object
etizolam_percent                   object
flualprazolam_percent              object
ketamine_percent                   object
kratom_percent                     object
synthetic_cannabinoid_percent      object
cathinones_percent                 object
aian_nh_percent                   

In [46]:
df_clean['ketamine_percent'].unique()

array(['0', '0.7', '0.2', '0.5', '0.3', '0.8', '1.3', '0.9', '0.4', '1.2',
       '0.6', '1.6', '1', '0.1', '1.1', '1.9', '2.3', '1.5', '1.4', '2'],
      dtype=object)

In [55]:
bb = df_clean.copy()
bb['ketamine_percent'] = df_clean['ketamine_percent'].astype(float)
bb.dtypes

Jurisdiction                        object
year                                 int64
alldrug_deaths                       int64
alldrug_rate                       float64
opioids_deaths                       int64
opioids_rate                       float64
imfs_deaths                          int64
imfs_rate                          float64
heroin_deaths                        int64
heroin_rate                        float64
rxopioids_deaths                     int64
rxopioids_rate                     float64
stimulant_deaths                     int64
stimulant_rate                     float64
cocaine_deaths                       int64
cocaine_rate                       float64
meth_deaths                          int64
meth_rate                          float64
nonopioid_sedatives_deaths           int64
nonopioid_sedatives_rate           float64
benzodiazepines_deaths               int64
benzodiazepines_rate               float64
opioids_percent                    float64
imfs_percen